# Municipal Inequality and Hardship in Denmark

[Back to website](../index.html)

**Research question:** How unequal are Danish municipalities, how has that changed over time, and why are the municipalities with the highest income inequality not always the same as the municipalities facing the strongest poverty and unemployment pressure?

This explainer notebook supports the final website. It contains the data work, validation checks, distributional analysis, housing-tenure bridge, visualization choices, and interpretation notes behind the public page.


## 1. Motivation

We write the website for a non-technical reader who wants to understand whether Danish inequality is only a national trend or also differs by municipality. We use a magazine-style sequence instead of a dashboard: national trend, municipal distribution, municipal map, ranked comparison, housing-tenure bridge, poverty-unemployment scatterplot, and an over-time robustness check. This follows Segel and Heer's idea that a data story can guide attention while still giving readers room to inspect details (2010).

The main claim is simple: municipal inequality is not the same as municipal hardship. Gini is useful for comparing income distributions, but the latest municipal Gini-poverty relationship is nearly flat, while poverty and unemployment move together much more strongly. Housing tenure helps explain where hardship concentrates: renter-heavy municipalities tend to have higher poverty and unemployment, while owner-heavy municipalities often sit lower on those hardship measures.

By the end, a general reader should be able to explain why a wealthy, high-Gini municipality can look very different from a high-poverty municipality, why housing tenure is a useful bridge but not a causal proof, why the map should not be read alone, and why we keep the claims descriptive rather than causal.


## 2. Dataset

The data comes from official Statistics Denmark StatBank tables:

- `IFOR41`: Gini, Hoover, S80/S20, and P90/P10 income-distribution indicators.
- `IFOR32`: average equivalised disposable income by decile.
- `IFOR12P`: persons in risk-of-poverty families at 50 percent and 60 percent thresholds.
- `AUP02`: monthly unemployed in percent of the labour force.
- `HFUDD11`: educational attainment for ages 15-69.
- `HISBK`: life expectancy for newborn babies.
- `BOL101`: dwellings with registered population by municipality and tenure.
- `BOLRD`: national dwellings with registered population by tenure, used to cross-check BOL101 tenure totals.

Municipality boundaries come from the public Dataforsyningen municipality GeoJSON endpoint. We use the checked-in local extract of the data tables, with APA-style source entries and retrieval dates listed in the references.

We chose these tables because they are official, municipality-level sources with enough overlap for the course workflow: inspect the raw schemas, clean and merge tabular data, check missingness, visualize distributions and relationships, and publish a one-page public story. The unit of analysis is municipality-year, with a national row kept for national trend plots and removed for municipal cross-sectional analysis. Housing tenure is matched only by same municipality-year, so newer 2026 housing values are not carried back into the 2024 income panel. BOLRD is not used for municipality-level joins because the StatBank extract has tenure and time but no municipality dimension.


## 3. Imports and project paths

Run the downloader first if raw data is missing:

```bash
python scripts/download_statbank_data.py
```

This pass uses the local raw data rather than fetching the official sources. The notebook can rebuild all processed files and visualizations from those raw CSVs.


In [1]:
from pathlib import Path
import json
import sys

import pandas as pd
from IPython.display import HTML, IFrame, display

pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 160)

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT))
from scripts.build_project import IMAGES_DIR, PROCESSED_DIR, VIS_DIR, run_pipeline


def relative_project_path(path):
    return str(Path(path).relative_to(PROJECT_ROOT)).replace("\\", "/")

## 4. Cleaning and preprocessing

The build script does the following:

- reads semicolon-separated StatBank CSVs with official codes and labels,
- splits combined code-label fields into explicit code and readable label columns,
- parses years, months, and rolling life-expectancy periods,
- pivots inequality and poverty indicators into a municipality-year panel,
- aggregates monthly unemployment into annual municipal means,
- calculates tertiary education share from education counts,
- aggregates BOL101 owner- and tenant-occupied dwellings into `OwnerShare` and `TenantShare`,
- cross-checks national BOL101 tenure totals against BOLRD,
- exports processed CSVs and website-ready visualizations.


In [2]:
outputs = run_pipeline(skip_figures=False)

output_rows = []
for path in outputs["processed_files"]:
    output_rows.append({"kind": "processed", "name": Path(path).name, "relative_path": relative_project_path(path)})
for name, path in outputs.get("figures", {}).items():
    output_rows.append({"kind": "figure", "name": name, "relative_path": relative_project_path(path)})

print(f"Panel rows: {outputs['panel_rows']:,}")
print(f"Decile rows: {outputs['decile_rows']:,}")
display(pd.DataFrame(output_rows))


Panel rows: 3,762
Decile rows: 37,620


,kind,name,relative_path
0,processed,municipality_year.csv,data/processed/municipality_year.csv
1,processed,decile_distribution.csv,data/processed/decile_distribution.csv
2,processed,missingness_summary.csv,data/processed/missingness_summary.csv
3,processed,regression_summary.csv,data/processed/regression_summary.csv
4,processed,correlation_over_time.csv,data/processed/correlation_over_time.csv
5,processed,housing_tenure_crosscheck.csv,data/processed/housing_tenure_crosscheck.csv
6,figure,trend,images/dk_inequality_trend.png
7,figure,deciles,images/dk_decile_distribution.png
8,figure,distribution,images/dk_municipal_distribution.png
9,figure,correlation_robustness,images/dk_correlation_robustness.png


## 5. Basic stats

We check that the data has the expected coverage: 99 geographies, 38 years in the core income tables, and a complete 2024 cross-section for the main inequality, poverty, unemployment, education, and housing-tenure measures. Life expectancy is present for most municipalities but is missing for four small island municipalities in the latest merged year.

We also add distribution and ranked tables for shape, spread, tails, and outliers. They show benchmark-year municipal distributions, the high-inequality/high-poverty contrast, and the housing-tenure bridge before we move into visual interpretation.


In [3]:
municipality_year = pd.read_csv(PROCESSED_DIR / "municipality_year.csv", dtype={"MunicipalityCode": str})
deciles = pd.read_csv(PROCESSED_DIR / "decile_distribution.csv", dtype={"MunicipalityCode": str})
missingness = pd.read_csv(PROCESSED_DIR / "missingness_summary.csv")
regression_summary = pd.read_csv(PROCESSED_DIR / "regression_summary.csv")
correlation_over_time = pd.read_csv(PROCESSED_DIR / "correlation_over_time.csv")
housing_crosscheck = pd.read_csv(PROCESSED_DIR / "housing_tenure_crosscheck.csv")
summary_stats = json.loads((PROCESSED_DIR / "summary_stats.json").read_text(encoding="utf-8"))

basic_stats = pd.DataFrame({
    "table": ["municipality_year", "decile_distribution", "housing_tenure_crosscheck"],
    "rows": [len(municipality_year), len(deciles), len(housing_crosscheck)],
    "columns": [municipality_year.shape[1], deciles.shape[1], housing_crosscheck.shape[1]],
})

year_coverage = (
    municipality_year.groupby("Year")
    .size()
    .rename("rows")
    .reset_index()
    .agg(first_year=("Year", "min"), last_year=("Year", "max"), years=("Year", "nunique"), rows=("rows", "sum"))
)

print("Basic table sizes")
display(basic_stats)
print("Municipality-year coverage")
display(year_coverage)
print("Missingness summary")
display(missingness)


Basic table sizes


,table,rows,columns
0,municipality_year,3762,17
1,decile_distribution,37620,7
2,housing_tenure_crosscheck,15,15


Municipality-year coverage


,Year,rows
first_year,1987.0,NaN
last_year,2024.0,NaN
years,38.0,NaN
rows,NaN,3762.0


Missingness summary


,Column,MissingPercent
0,Year,0.0
1,MunicipalityCode,0.0
2,Municipality,0.0
3,Gini,0.0
4,P90P10,0.0
5,S80S20,0.0
6,Poverty50,34.2
7,Poverty60,34.2
8,UnemploymentRate,52.6
9,TertiaryShare,55.3


In [4]:
municipal = municipality_year[municipality_year["MunicipalityCode"] != "000"].copy()

range_checks = []
for column, low, high in [
    ("Gini", 0, 100),
    ("P90P10", 0, None),
    ("S80S20", 0, None),
    ("Poverty50", 0, 100),
    ("Poverty60", 0, 100),
    ("UnemploymentRate", 0, 100),
    ("TertiaryShare", 0, 100),
    ("LifeExpectancy", 0, 120),
    ("OwnerShare", 0, 100),
    ("TenantShare", 0, 100),
]:
    series = municipality_year[column].dropna()
    bad = series[series < low]
    if high is not None:
        bad = series[(series < low) | (series > high)]
    range_checks.append({
        "column": column,
        "min": series.min(),
        "max": series.max(),
        "outside_expected_range": len(bad),
    })

validation = pd.DataFrame(range_checks)
latest_year = int(municipality_year["Year"].max())
latest_life_missing = municipal[
    (municipal["Year"] == latest_year) & municipal["LifeExpectancy"].isna()
][["MunicipalityCode", "Municipality"]]
negative_s80s20 = municipal[municipal["S80S20"] < 0][
    ["Year", "MunicipalityCode", "Municipality", "S80S20"]
]
latest_housing = municipal[
    (municipal["Year"] == latest_year) & municipal["OwnerShare"].notna()
].copy()
housing_share_sum = latest_housing["OwnerShare"] + latest_housing["TenantShare"]
zero_housing_denominators = int((latest_housing["KnownTenureDwellings"] <= 0).sum())

print("Duplicate municipality-year keys:", municipality_year.duplicated(["MunicipalityCode", "Year"]).sum())
display(validation)
print("Latest-year missing life expectancy")
display(latest_life_missing)
print("Source-level negative S80/S20 caveat")
display(negative_s80s20)
print(f"Latest-year housing rows: {len(latest_housing)}")
print(f"Zero housing denominators: {zero_housing_denominators}")
print(f"OwnerShare + TenantShare range: {housing_share_sum.min():.6f} to {housing_share_sum.max():.6f}")


Duplicate municipality-year keys: 0


,column,min,max,outside_expected_range
0,Gini,15.900000,54.160000,0
1,P90P10,1.960000,6.160000,0
2,S80S20,-12.730000,21.020000,1
3,Poverty50,1.900000,16.400000,0
4,Poverty60,3.100000,24.800000,0
5,UnemploymentRate,1.041667,10.833333,0
6,TertiaryShare,13.759376,58.957461,0
7,LifeExpectancy,74.600000,83.800000,0
8,OwnerShare,0.379054,80.791071,0
9,TenantShare,19.208929,99.620946,0


Latest-year missing life expectancy


,MunicipalityCode,Municipality
2165,492,Ærø
2393,563,Fanø
3077,741,Samsø
3571,825,Læsø


Source-level negative S80/S20 caveat


,Year,MunicipalityCode,Municipality,S80S20
919,1994,230,Rudersdal,-12.73


Latest-year housing rows: 98
Zero housing denominators: 0
OwnerShare + TenantShare range: 100.000000 to 100.000000


### Housing tenure source check

`BOL101` remains the housing source for municipal joins because it includes the region/municipality dimension. `BOLRD` has tenure and time only in the StatBank extract, so we use it as a national quality-control check for the owner/tenant totals rather than as a replacement panel.


In [5]:
housing_check_columns = [
    "Year",
    "BOL101OwnerOccupiedDwellings",
    "BOLRDOwnerOccupiedDwellings",
    "OwnerOccupiedDwellingsDiff",
    "BOL101TenantOccupiedDwellings",
    "BOLRDTenantOccupiedDwellings",
    "TenantOccupiedDwellingsDiff",
    "BOL101OwnerShare",
    "BOLRDOwnerShare",
    "OwnerShareDiffPp",
    "Status",
]
latest_housing_crosscheck = housing_crosscheck.sort_values("Year").tail(1)
print("Latest BOL101 vs BOLRD national tenure cross-check")
display(latest_housing_crosscheck[housing_check_columns].round({
    "BOL101OwnerShare": 3,
    "BOLRDOwnerShare": 3,
    "OwnerShareDiffPp": 4,
}))
print("Max absolute owner-share difference, percentage points:", round(housing_crosscheck["OwnerShareDiffPp"].abs().max(), 4))


Latest BOL101 vs BOLRD national tenure cross-check


,Year,BOL101OwnerOccupiedDwellings,BOLRDOwnerOccupiedDwellings,OwnerOccupiedDwellingsDiff,BOL101TenantOccupiedDwellings,BOLRDTenantOccupiedDwellings,TenantOccupiedDwellingsDiff,BOL101OwnerShare,BOLRDOwnerShare,OwnerShareDiffPp,Status
14,2026,1355272,1355272.0,0.0,1496878,1496878.0,0.0,47.518,47.518,0.0,ok


Max absolute owner-share difference, percentage points: 0.0


In [6]:
distribution_stats = pd.DataFrame(summary_stats["benchmark_distribution_stats"])
print("Benchmark municipal distribution stats")
display(distribution_stats[["metric", "year", "count", "min", "q1", "median", "q3", "max"]].round(2))

rank_year = summary_stats["latest_rank_summary"]["year"]
top_gini = pd.DataFrame(summary_stats["latest_rank_summary"]["top_gini"])
top_poverty = pd.DataFrame(summary_stats["latest_rank_summary"]["top_poverty60"])
housing = summary_stats["housing_summary"]
low_owner = pd.DataFrame(housing["lowest_owner_share"])
high_owner = pd.DataFrame(housing["highest_owner_share"])

print(f"Latest ranked comparison year: {rank_year}")
print("Highest-Gini municipalities")
display(top_gini.round({"Gini": 2, "Poverty60": 1, "UnemploymentRate": 2}))
print("Highest-poverty municipalities")
display(top_poverty.round({"Gini": 2, "Poverty60": 1, "UnemploymentRate": 2}))
print(f"Housing summary year: {housing['year']}")
print("Lowest owner-share municipalities")
display(low_owner.round({"OwnerShare": 1, "TenantShare": 1, "Poverty60": 1, "UnemploymentRate": 2, "Gini": 2}))
print("Highest owner-share municipalities")
display(high_owner.round({"OwnerShare": 1, "TenantShare": 1, "Poverty60": 1, "UnemploymentRate": 2, "Gini": 2}))


Benchmark municipal distribution stats


,metric,year,count,min,q1,median,q3,max
0,Gini,1987,98,15.90,20.09,20.88,21.56,31.10
1,Gini,2000,98,19.58,21.34,22.08,22.84,37.57
2,Gini,2010,98,21.65,23.73,24.74,25.78,43.42
3,Gini,2024,98,23.77,25.40,26.40,27.36,49.04
4,Poverty60,2000,98,3.30,7.02,8.50,9.78,20.40
5,Poverty60,2010,98,4.50,8.70,9.85,11.28,23.50
6,Poverty60,2024,98,5.40,9.65,11.35,13.30,20.60


Latest ranked comparison year: 2024
Highest-Gini municipalities


,Municipality,Gini,Poverty60,UnemploymentRate
0,Gentofte,49.04,9.6,2.33
1,Rudersdal,47.31,7.6,1.97
2,Vejen,46.32,10.5,1.86
3,Hørsholm,40.07,7.3,2.15
4,Lyngby-Taarbæk,39.28,11.8,2.21
5,Frederiksberg,36.54,14.9,3.20
6,Copenhagen,34.97,20.6,3.58
7,Aarhus,33.43,20.1,3.68


Highest-poverty municipalities


,Municipality,Gini,Poverty60,UnemploymentRate
0,Copenhagen,34.97,20.6,3.58
1,Aarhus,33.43,20.1,3.68
2,Ærø,26.98,18.6,2.01
3,Odense,28.32,18.4,3.89
4,Lolland,26.45,18.2,3.48
5,Brøndby,26.16,17.4,3.78
6,Langeland,25.27,17.1,3.55
7,Ishøj,24.88,16.6,4.68


Housing summary year: 2024
Lowest owner-share municipalities


,Municipality,OwnerShare,TenantShare,Poverty60,UnemploymentRate,Gini
0,Copenhagen,19.3,80.7,20.6,3.58,34.97
1,Frederiksberg,22.7,77.3,14.9,3.20,36.54
2,Brøndby,26.9,73.1,17.4,3.78,26.16
3,Herlev,31.4,68.6,11.3,2.47,27.15
4,Albertslund,32.5,67.5,15.8,3.26,24.66
5,Ballerup,33.0,67.0,11.1,2.34,26.72
6,Aarhus,33.9,66.1,20.1,3.68,33.43
7,Glostrup,35.6,64.4,12.9,2.86,26.20


Highest owner-share municipalities


,Municipality,OwnerShare,TenantShare,Poverty60,UnemploymentRate,Gini
0,Lejre,75.9,24.1,7.4,2.08,25.36
1,Læsø,74.1,25.9,14.3,4.72,25.41
2,Dragør,72.6,27.4,5.7,1.69,28.86
3,Gribskov,72.2,27.8,9.6,1.96,27.68
4,Egedal,71.8,28.2,6.1,1.75,23.85
5,Allerød,70.3,29.7,5.4,1.51,27.29
6,Rebild,69.5,30.5,7.6,2.10,25.87
7,Favrskov,68.6,31.4,7.4,2.27,24.41


## 6. Data analysis

We keep the analysis descriptive. We use distribution summaries, Pearson-style correlations, and fitted lines to move from exploration toward explanation without making causal claims.

In 2024, the clearest relationship is between the 60 percent risk-of-poverty rate and unemployment. The Poverty60-unemployment relationship has `R2 = .529` across 98 municipalities. The latest Gini-poverty relationship is close to flat at `R2 = .002`, so we use Gini as an inequality measure and poverty/unemployment as the stronger hardship indicators.

Housing tenure adds a useful bridge. OwnerShare is negatively associated with Poverty60 (`R2 = .304`) and unemployment (`R2 = .220`), while OwnerShare and Gini are only weakly related (`R2 = .029`). This supports the narrative that renter-heavy municipalities often carry more hardship, but it does not turn tenure into a causal explanation.

We also look at why the inequality and poverty lists differ. In 2024, Gini is more closely associated with tertiary education share and life expectancy than with poverty or unemployment. That pattern fits the idea that some high-Gini municipalities are shaped by high-income tails. Poverty60, by contrast, is negatively associated with life expectancy, strongly associated with unemployment, and more common in renter-heavy municipalities.

We repeat the two core cross-sectional relationships year by year. This supports the time dimension of the research question and shows that the project is not relying only on the 2024 endpoint. Readers can see the distributions, ranked lists, points, fitted line, R2 value, and limits of the interpretation directly.


In [7]:
print("Latest-year regression summary")
display(regression_summary.round({"slope": 3, "intercept": 3, "r": 3, "r2": 3, "p_value": 4}))

selected = regression_summary[(regression_summary["x"] == "Poverty60") & (regression_summary["y"] == "UnemploymentRate")].iloc[0]
weak = regression_summary[(regression_summary["x"] == "Gini") & (regression_summary["y"] == "Poverty60")].iloc[0]
housing_relationships = regression_summary[regression_summary["x"] == "OwnerShare"].copy()
housing_relationships["relationship"] = housing_relationships["x"] + " vs " + housing_relationships["y"]

why_lookup = {
    ("Gini", "TertiaryShare"): "High-Gini places also tend to have larger tertiary-educated populations.",
    ("Gini", "LifeExpectancy"): "High-Gini places are not necessarily high-hardship places in this dataset.",
    ("Poverty60", "LifeExpectancy"): "Poverty is more clearly tied to a hardship-related health context.",
    ("OwnerShare", "Poverty60"): "Renter-heavy municipalities tend to have higher poverty.",
    ("OwnerShare", "UnemploymentRate"): "Renter-heavy municipalities also tend to have higher unemployment.",
    ("OwnerShare", "Gini"): "Housing tenure is much less aligned with Gini than with hardship.",
}
why_rows = regression_summary[
    regression_summary[["x", "y"]].apply(tuple, axis=1).isin(why_lookup.keys())
].copy()
why_rows["relationship"] = why_rows["x"] + " vs " + why_rows["y"]
why_rows["interpretation"] = why_rows[["x", "y"]].apply(lambda row: why_lookup[(row["x"], row["y"])], axis=1)

print("Why the inequality and hardship lists differ")
display(
    why_rows[["Year", "relationship", "n", "r", "r2", "p_value", "interpretation"]]
    .round({"r": 3, "r2": 3, "p_value": 4})
)
print("Housing tenure relationships")
display(housing_relationships[["Year", "relationship", "n", "slope", "r", "r2", "p_value"]].round({"slope": 3, "r": 3, "r2": 3, "p_value": 4}))

latest = municipal[municipal["Year"] == summary_stats["latest_rank_summary"]["year"]].copy()
robustness_pairs = [
    ("Gini", "Poverty60"),
    ("P90P10", "Poverty60"),
    ("S80S20", "Poverty60"),
    ("Gini", "Poverty50"),
    ("OwnerShare", "Poverty60"),
    ("OwnerShare", "UnemploymentRate"),
    ("OwnerShare", "Gini"),
    ("Poverty50", "UnemploymentRate"),
    ("Poverty60", "UnemploymentRate"),
]
robustness_rows = []
for x, y in robustness_pairs:
    usable = latest.dropna(subset=[x, y])
    r = usable[x].corr(usable[y]) if len(usable) >= 3 else pd.NA
    robustness_rows.append({
        "x": x,
        "y": y,
        "n": len(usable),
        "r": r,
        "r2": r ** 2 if pd.notna(r) else pd.NA,
    })
robustness = pd.DataFrame(robustness_rows)

time_r2 = correlation_over_time.pivot(index="Year", columns="relationship", values="r2")

print(f"Poverty60 vs unemployment R2: {selected['r2']:.3f}")
print(f"Gini vs Poverty60 R2: {weak['r2']:.3f}")
print("Latest-year robustness checks")
display(robustness.round({"r": 3, "r2": 3}))
print("Recent over-time R2 checks")
display(time_r2.tail(8).round(3))


Latest-year regression summary


,Year,x,y,n,slope,intercept,r,r2,p_value
0,2024,Gini,Poverty60,98,-0.031,12.466,-0.045,0.002,0.6579
1,2024,Poverty60,UnemploymentRate,98,0.143,0.975,0.727,0.529,0.0000
2,2024,Poverty60,LifeExpectancy,94,-0.202,83.836,-0.627,0.393,0.0000
3,2024,Gini,UnemploymentRate,98,-0.022,3.232,-0.164,0.027,0.1077
4,2024,Gini,TertiaryShare,98,1.167,0.867,0.587,0.345,0.0000
5,2024,Gini,LifeExpectancy,94,0.099,78.777,0.466,0.217,0.0000
6,2024,OwnerShare,Poverty60,98,-0.148,19.878,-0.551,0.304,0.0000
7,2024,OwnerShare,UnemploymentRate,98,-0.025,4.008,-0.469,0.220,0.0000
8,2024,OwnerShare,Gini,98,-0.067,31.332,-0.169,0.029,0.0965


Why the inequality and hardship lists differ


,Year,relationship,n,r,r2,p_value,interpretation
2,2024,Poverty60 vs LifeExpectancy,94,-0.627,0.393,0.0000,Poverty is more clearly tied to a hardship-related health context.
4,2024,Gini vs TertiaryShare,98,0.587,0.345,0.0000,High-Gini places also tend to have larger tertiary-educated populations.
5,2024,Gini vs LifeExpectancy,94,0.466,0.217,0.0000,High-Gini places are not necessarily high-hardship places in this dataset.
6,2024,OwnerShare vs Poverty60,98,-0.551,0.304,0.0000,Renter-heavy municipalities tend to have higher poverty.
7,2024,OwnerShare vs UnemploymentRate,98,-0.469,0.220,0.0000,Renter-heavy municipalities also tend to have higher unemployment.
8,2024,OwnerShare vs Gini,98,-0.169,0.029,0.0965,Housing tenure is much less aligned with Gini than with hardship.


Housing tenure relationships


,Year,relationship,n,slope,r,r2,p_value
6,2024,OwnerShare vs Poverty60,98,-0.148,-0.551,0.304,0.0000
7,2024,OwnerShare vs UnemploymentRate,98,-0.025,-0.469,0.220,0.0000
8,2024,OwnerShare vs Gini,98,-0.067,-0.169,0.029,0.0965


Poverty60 vs unemployment R2: 0.529
Gini vs Poverty60 R2: 0.002
Latest-year robustness checks


,x,y,n,r,r2
0,Gini,Poverty60,98,-0.045,0.002
1,P90P10,Poverty60,98,0.060,0.004
2,S80S20,Poverty60,98,-0.046,0.002
3,Gini,Poverty50,98,0.130,0.017
4,OwnerShare,Poverty60,98,-0.551,0.304
5,OwnerShare,UnemploymentRate,98,-0.469,0.220
6,OwnerShare,Gini,98,-0.169,0.029
7,Poverty50,UnemploymentRate,98,0.644,0.415
8,Poverty60,UnemploymentRate,98,0.727,0.529


Recent over-time R2 checks


relationship,Gini vs Poverty60,Poverty60 vs UnemploymentRate
Year,,
2017,0.017,0.503
2018,0.018,0.514
2019,0.007,0.584
2020,0.011,0.483
2021,0.003,0.427
2022,0.011,0.450
2023,0.000,0.463
2024,0.002,0.529


## 7. Visualizations

We use seven focused charts to create a short data story with both static and interactive views.

1. **National trend**: a static matplotlib figure gives the long-run context and avoids overwhelming the first view with municipal detail.
2. **Municipal distribution**: a static box-and-jitter figure shows spread, tails, and benchmark-year movement across municipalities. This is better than another line chart because the claim depends on distributional shape, not only averages.
3. **Municipal inequality map**: an interactive Plotly choropleth shows geography and allows hover inspection. The map is not treated as the main proof because color maps are weaker for precise ranking.
4. **Inequality-poverty ranked comparison**: an interactive Plotly view makes the central contrast precise by putting high-Gini and high-poverty municipalities side by side.
5. **Housing-tenure bridge**: an interactive two-part Plotly view shows owner-share geography and the OwnerShare-Poverty60 relationship, colored by unemployment.
6. **Poverty-unemployment scatter**: an interactive Plotly scatterplot with a fitted line shows the 2024 poverty-unemployment relationship and lets readers inspect named municipalities.
7. **Over-time robustness**: a static R2 line chart checks whether the central contrast also appears outside 2024.

A national decile chart is still generated as supporting notebook material, but it is not embedded on the public page because the page focuses on municipal distributions and the difference between inequality and hardship. We also avoid putting every chart into one control-heavy page: it would offer more controls, but it would make the argument less direct.


In [8]:
display(HTML('<img src="../images/dk_inequality_trend.png" alt="Indexed trend chart for Danish inequality and poverty measures" style="max-width: 100%; height: auto;">'))


In [9]:
display(HTML('<img src="../images/dk_municipal_distribution.png" alt="Municipal Gini and poverty distribution plots" style="max-width: 100%; height: auto;">'))


In [10]:
display(IFrame(src="../visualizations/dk_inequality_map.html", width="100%", height=900))


In [11]:
display(IFrame(src="../visualizations/dk_inequality_poverty_rank.html", width="100%", height=650))


In [12]:
display(IFrame(src="../visualizations/dk_housing_tenure_bridge.html", width="100%", height=760))


In [13]:
display(IFrame(src="../visualizations/dk_poverty_unemployment_scatter.html", width="100%", height=650))


In [14]:
display(HTML('<img src="../images/dk_correlation_robustness.png" alt="Over-time R squared robustness chart" style="max-width: 100%; height: auto;">'))

visualization_files = pd.DataFrame({
    "file": [
        IMAGES_DIR / "dk_inequality_trend.png",
        IMAGES_DIR / "dk_municipal_distribution.png",
        IMAGES_DIR / "dk_correlation_robustness.png",
        VIS_DIR / "dk_inequality_map.html",
        VIS_DIR / "dk_inequality_poverty_rank.html",
        VIS_DIR / "dk_housing_tenure_bridge.html",
        VIS_DIR / "dk_poverty_unemployment_scatter.html",
    ]
})
visualization_files["exists"] = visualization_files["file"].map(lambda p: p.exists())
visualization_files["size_mb"] = visualization_files["file"].map(lambda p: round(p.stat().st_size / 1024 / 1024, 2) if p.exists() else None)
visualization_files["relative_path"] = visualization_files["file"].map(relative_project_path)
display(visualization_files[["relative_path", "exists", "size_mb"]])


,relative_path,exists,size_mb
0,images/dk_inequality_trend.png,True,0.13
1,images/dk_municipal_distribution.png,True,0.12
2,images/dk_correlation_robustness.png,True,0.08
3,visualizations/dk_inequality_map.html,True,0.36
4,visualizations/dk_inequality_poverty_rank.html,True,0.01
5,visualizations/dk_housing_tenure_bridge.html,True,0.15
6,visualizations/dk_poverty_unemployment_scatter.html,True,0.02


## 8. Genre

We use the **magazine-style narrative visualization** genre from Segel and Heer (2010). This fits the project because we have a specific argument to walk through: national inequality rose, municipal distributions shifted, geography matters, housing tenure helps explain local hardship patterns, and Gini alone does not capture hardship.

Using Segel and Heer's Figure 7 vocabulary, the visual narrative choices are:

- **Visual structuring**: one linear page, repeated figure/caption rhythm, and consistent typography/color across the static and interactive charts.
- **Highlighting**: the headline states the main contrast, the distribution plot makes spread visible, the ranked comparison names the contrast, the housing bridge explains where hardship concentrates, the scatterplot shows how poverty and unemployment move together, and the robustness figure checks the endpoint choice.
- **Transition guidance**: section headings and captions move the reader from national trend to distribution, geography, precise ranking, housing tenure, poverty-unemployment association, and over-time check.

The narrative-structure choices are:

- **Ordering**: a linear magazine sequence, not a dashboard.
- **Interactivity**: hover labels in the map, ranking, housing, and scatterplot, while keeping the argument fixed.
- **Messaging**: headline, deck, glossary, figure captions, takeaways, limits, and APA-style references make the claim and caveats explicit.


## 9. Discussion

The main strength is the data base. It is official, reproducible, and local enough for municipal comparison. The seven-chart structure gives enough evidence for the main claim without turning the site into a dashboard. The distribution figure examines shape and spread, the ranked comparison makes the difference between inequality and hardship concrete, the housing-tenure bridge gives that difference local structure, and the over-time R2 figure checks that the conclusion is not only about the endpoint year.

The added housing explanation helps make sense of the split between the lists. We still do not claim that education, life expectancy, poverty, unemployment, or housing tenure causes municipal inequality. Instead, we show that some high-Gini places also have higher education shares and longer life expectancy, while high-poverty places often face more unemployment and have lower owner-occupancy. That helps explain why Gentofte or Rudersdal should not be read the same way as Lolland, Ishoj, Copenhagen, or Aarhus, even though all are important to the story.

The limits are important. We cannot identify causal effects, separate household wealth from income, or measure housing costs and consumption. BOL101 gives municipal tenure, not rent burden or local housing prices, and 2021-2022 are closed in the source because of BBR data errors. BOLRD validates the national owner/tenant totals, but it does not add a municipality dimension. Life expectancy is measured over rolling periods and is noisier in small municipalities. The latest merged income panel ends in 2024 even though housing has 2026 observations; we do not carry those newer housing values backward. A future version could add population weighting, housing-cost indicators, and a formal panel model, but those additions would answer a more causal question than this course project is meant to answer.


## 10. Contributions

This is treated as a solo project. The same author was responsible for data collection, cleaning, validation, visualization generation, notebook writing, and website assembly.


## 11. References

- Dataforsyningen. (n.d.). *Kommuner* [GeoJSON data set]. Retrieved May 12, 2026, from <https://api.dataforsyningen.dk/kommuner?format=geojson>
- Segel, E., & Heer, J. (2010). Narrative visualization: Telling stories with data. *IEEE Transactions on Visualization and Computer Graphics, 16*(6), 1139-1148. <https://doi.org/10.1109/TVCG.2010.179>
- Statistics Denmark. (n.d.). *Average equivalised disposable income by decile (IFOR32)* [Data set]. StatBank Denmark. Retrieved May 12, 2026, from <https://www.statbank.dk/IFOR32>
- Statistics Denmark. (n.d.). *Dwellings (BOL101)* [Data set]. StatBank Denmark. Retrieved May 13, 2026, from <https://www.statbank.dk/BOL101>
- Statistics Denmark. (n.d.). *Dwellings with registered population by tenure (BOLRD)* [Data set]. StatBank Denmark. Retrieved May 13, 2026, from <https://www.statbank.dk/BOLRD>
- Statistics Denmark. (n.d.). *Educational attainment (HFUDD11)* [Data set]. StatBank Denmark. Retrieved May 12, 2026, from <https://www.statbank.dk/HFUDD11>
- Statistics Denmark. (n.d.). *Income distribution on equivalised disposable income (IFOR41)* [Data set]. StatBank Denmark. Retrieved May 12, 2026, from <https://www.statbank.dk/IFOR41>
- Statistics Denmark. (n.d.). *Life expectancy for newborn babies (HISBK)* [Data set]. StatBank Denmark. Retrieved May 12, 2026, from <https://www.statbank.dk/HISBK>
- Statistics Denmark. (n.d.). *Persons in risk-of-poverty families (IFOR12P)* [Data set]. StatBank Denmark. Retrieved May 12, 2026, from <https://www.statbank.dk/IFOR12P>
- Statistics Denmark. (n.d.). *Unemployed in percent of the labour force (AUP02)* [Data set]. StatBank Denmark. Retrieved May 12, 2026, from <https://www.statbank.dk/AUP02>
